In [ ]:
# ЯЧЕЙКА 1: только проверка интернет-канала (не блокирует работу ноутбука).
import os  # Импорт os нужен для чтения переменных окружения (если пользователь захочет переопределить пороги).
import re  # Импорт re нужен для извлечения среднего ping из текстового вывода команды ping.
import subprocess  # Импорт subprocess нужен для запуска системной команды ping.

MIN_DOWNLOAD_MBPS = float(os.environ.get("MIN_DOWNLOAD_MBPS", "300"))  # Константа задаёт минимально желаемую скорость скачивания в Мбит/с.
MAX_PING_MS = float(os.environ.get("MAX_PING_MS", "100"))  # Константа задаёт максимально желаемый средний ping в миллисекундах.
PING_HOSTS = ["civitai.com", "huggingface.co"]  # Список сайтов для проверки ping (критерий ping остаётся прежним).


def ensure_speedtest_package():
    """Функция проверяет наличие speedtest и устанавливает пакет, если он отсутствует."""
    import importlib.util as importlib_util  # Локальный импорт для проверки существования Python-модуля.
    module_exists = importlib_util.find_spec("speedtest") is not None  # Флаг показывает, установлен ли модуль speedtest.
    if module_exists:
        print("speedtest-cli: уже установлен")
        return True
    print("speedtest-cli: не найден, устанавливаю...")
    install_proc = subprocess.run(  # Результат выполнения pip install сохраняется в переменную install_proc.
        ["python", "-m", "pip", "install", "-U", "speedtest-cli"],
        text=True,
        capture_output=True,
    )
    if install_proc.returncode != 0:  # Если код возврата не нулевой, установка не удалась.
        print("[WARN] Не удалось установить speedtest-cli. Проверка скорости будет пропущена.")
        print(install_proc.stdout[-800:])
        print(install_proc.stderr[-800:])
        return False
    return True


def run_speedtest():
    """Функция измеряет download/upload скорость и возвращает словарь результатов."""
    import speedtest  # Импорт внутри функции, чтобы не ломать весь ноутбук, если пакет не установлен.
    result = {"download_mbps": None, "upload_mbps": None, "error": None}  # Словарь result хранит результат теста сети.
    try:
        st = speedtest.Speedtest(secure=True)  # Объект st выполняет подбор сервера и сами измерения.
        st.get_best_server()
        download_bps = st.download()  # Скорость скачивания в битах в секунду.
        upload_bps = st.upload()  # Скорость отдачи в битах в секунду.
        result["download_mbps"] = download_bps / 1_000_000  # Перевод в Мбит/с.
        result["upload_mbps"] = upload_bps / 1_000_000  # Перевод в Мбит/с.
    except Exception as exc:
        result["error"] = str(exc)  # В поле error сохраняется причина сбоя speedtest.
    return result


def avg_ping(host):
    """Функция запускает ping до host и возвращает среднее значение в мс."""
    cmd = ["ping", "-c", "3", "-W", "2", host]  # Команда ping: 3 пакета и таймаут 2 секунды на ответ.
    ping_proc = subprocess.run(cmd, text=True, capture_output=True)  # Результат ping сохраняем в ping_proc.
    if ping_proc.returncode != 0:
        return None, ping_proc.stderr.strip() or ping_proc.stdout.strip()  # Возвращаем ошибку, если ping неуспешен.
    summary_match = re.search(r"=\s*([\d\.]+)/([\d\.]+)/([\d\.]+)/", ping_proc.stdout)  # Поиск строки min/avg/max.
    if not summary_match:
        return None, "Не удалось распарсить средний ping"
    avg_ms = float(summary_match.group(2))  # Второе число в шаблоне — это средний ping (avg).
    return avg_ms, None


print("Проверка интернета (требования: download >= 300 Мбит/с и ping < 100 мс)...")
can_run_speedtest = ensure_speedtest_package()  # Флаг показывает, доступен ли speedtest для измерения скорости.
net = {"download_mbps": None, "upload_mbps": None, "pings": {}}  # Словарь net хранит итоговые сетевые метрики.

if can_run_speedtest:
    speed_result = run_speedtest()  # Переменная speed_result хранит результат speedtest.
    net["download_mbps"] = speed_result.get("download_mbps")
    net["upload_mbps"] = speed_result.get("upload_mbps")
    if speed_result.get("error"):
        print(f"[WARN] Ошибка speedtest: {speed_result['error']}")
else:
    print("[WARN] Speedtest пропущен (модуль недоступен).")

for host in PING_HOSTS:
    host_ping, host_err = avg_ping(host)  # host_ping — средний ping, host_err — текст ошибки.
    net["pings"][host] = {"avg_ms": host_ping, "error": host_err}

print("\n--- Результаты сети ---")
print(f"Download: {net['download_mbps']:.2f} Мбит/с" if net["download_mbps"] is not None else "Download: N/A")
print(f"Upload:   {net['upload_mbps']:.2f} Мбит/с" if net["upload_mbps"] is not None else "Upload:   N/A")
for host, host_data in net["pings"].items():
    print(f"Ping {host}: {host_data['avg_ms']:.2f} мс" if host_data["avg_ms"] is not None else f"Ping {host}: N/A ({host_data['error']})")

speed_ok = net["download_mbps"] is not None and net["download_mbps"] >= MIN_DOWNLOAD_MBPS  # Проверка нового критерия скорости (>=300).
ping_ok = all(host_data["avg_ms"] is not None and host_data["avg_ms"] < MAX_PING_MS for host_data in net["pings"].values())  # Проверка старого критерия ping.

if speed_ok and ping_ok:
    print("✅ Сеть соответствует требованиям для загрузок и работы.")
else:
    print("⚠️ Предупреждение: сеть ниже рекомендуемого уровня (download < 300 Мбит/с и/или ping >= 100 мс).")


In [ ]:
# ЯЧЕЙКА 2: базовые переменные окружения и директории только для ComfyUI на Vast.ai.
import os  # Импорт os нужен для работы с переменными окружения.
from pathlib import Path  # Импорт Path нужен для понятной и безопасной работы с путями.

ON_VAST = any(key in os.environ for key in ("VAST_CONTAINERLABEL", "VAST_TCP_PORT_22", "CONTAINER_ID")) or Path("/workspace").exists()  # Флаг окружения Vast.ai.
if not ON_VAST:
    print("⚠️ Внимание: окружение не похоже на Vast.ai, но код продолжит работу.")

BASE_DIR = Path(os.environ.get("BASE_DIR", "/workspace"))  # Корневая рабочая директория.
COMFY_DIR = BASE_DIR / "ComfyUI"  # Путь к репозиторию ComfyUI.
VOLUME_DIR = Path(os.environ.get("VOLUME_DIR", str(BASE_DIR / "volume")))  # Путь к постоянному тому (volume).

MODELS_DIR = COMFY_DIR / "models"  # Общая папка моделей ComfyUI.
CHECKPOINTS_DIR = MODELS_DIR / "checkpoints"  # Папка базовых моделей (checkpoints).
LORA_DIR = MODELS_DIR / "loras"  # Папка LoRA моделей.
UPSCALER_DIR = MODELS_DIR / "upscale_models"  # Папка upscaler моделей.
VAE_DIR = MODELS_DIR / "vae"  # Папка VAE.
CONTROLNET_DIR = MODELS_DIR / "controlnet"  # Папка ControlNet моделей.
IPADAPTER_DIR = MODELS_DIR / "ipadapter"  # Папка IP-Adapter моделей.
CUSTOM_NODES_DIR = COMFY_DIR / "custom_nodes"  # Папка для custom nodes ComfyUI.

SECRET_ENV_NAMES = {  # Словарь только для отображения факта наличия секретов (ввод в ноутбуке не нужен).
    "CIVITAI_TOKEN": os.environ.get("CIVITAI_TOKEN", ""),
    "HF_TOKEN": os.environ.get("HF_TOKEN", ""),
    "GRADIO_TOKEN": os.environ.get("GRADIO_TOKEN", ""),
}

for required_dir in [BASE_DIR, COMFY_DIR, VOLUME_DIR, MODELS_DIR, CHECKPOINTS_DIR, LORA_DIR, UPSCALER_DIR, VAE_DIR, CONTROLNET_DIR, IPADAPTER_DIR, CUSTOM_NODES_DIR]:
    required_dir.mkdir(parents=True, exist_ok=True)  # Создаём директории заранее, чтобы следующие ячейки работали стабильно.

print("✅ Базовые переменные и директории ComfyUI подготовлены.")
print(f"COMFY_DIR: {COMFY_DIR}")
print(f"VOLUME_DIR: {VOLUME_DIR}")


In [ ]:
# ЯЧЕЙКА 3: установка зависимостей и ComfyUI (логика: проверка -> установка при отсутствии).
import subprocess  # Импорт subprocess нужен для запуска apt/pip/git команд.
from pathlib import Path  # Импорт Path нужен для проверки наличия папок/файлов.

COMFY_REPO_URL = "https://github.com/comfyanonymous/ComfyUI.git"  # URL официального репозитория ComfyUI.
APT_PACKAGES = ["git", "python3-venv", "python3-pip", "aria2", "libgl1", "libglib2.0-0"]  # Системные пакеты для работы и загрузок.
PYTHON_PACKAGES = ["requests", "tqdm", "huggingface_hub", "gdown", "gradio"]  # Python-пакеты, нужные в ячейках загрузки и запуска.


def run_cmd(cmd, check=True):
    """Функция запускает команду и печатает её, чтобы действия были прозрачны для пользователя."""
    print("[CMD]", " ".join(str(x) for x in cmd))
    return subprocess.run(cmd, check=check)


run_cmd(["apt", "update", "-qq"], check=False)
run_cmd(["apt", "install", "-y", "-qq", *APT_PACKAGES], check=False)
run_cmd(["python", "-m", "pip", "install", "-U", "pip"], check=False)
run_cmd(["python", "-m", "pip", "install", "-U", *PYTHON_PACKAGES], check=False)

comfy_git_dir = Path(COMFY_DIR) / ".git"  # Путь .git нужен для понимания: это уже git-репозиторий или нет.
if comfy_git_dir.exists():
    print("ComfyUI уже существует, выполняю git pull...")
    run_cmd(["git", "-C", str(COMFY_DIR), "pull", "--ff-only"], check=False)
else:
    print("ComfyUI не найден, выполняю git clone...")
    run_cmd(["git", "clone", COMFY_REPO_URL, str(COMFY_DIR)], check=True)

requirements_file = Path(COMFY_DIR) / "requirements.txt"  # Путь к файлу зависимостей ComfyUI.
if requirements_file.exists():
    run_cmd(["python", "-m", "pip", "install", "-U", "-r", str(requirements_file)], check=False)

print("✅ Ячейка 3 завершена: зависимости и ComfyUI готовы.")


In [ ]:
# ЯЧЕЙКА 4: формируем списки и выполняем параллельные загрузки (до 5 потоков, приоритет: тяжёлые -> лёгкие).
import os  # Импорт os нужен для чтения токенов и настроек из переменных окружения.
import subprocess  # Импорт subprocess нужен для git/aria2c команд.
from pathlib import Path  # Импорт Path нужен для работы с путями и создания папок.
from concurrent.futures import ThreadPoolExecutor, as_completed  # Импорт для параллельных скачиваний.

MAX_PARALLEL_DOWNLOADS = int(os.environ.get("MAX_PARALLEL_DOWNLOADS", "5"))  # Максимум параллельных загрузок (расширено до 5).
MIN_VALID_FILE_BYTES = int(os.environ.get("MIN_VALID_FILE_BYTES", "1024"))  # Минимальный валидный размер файла.
CIVITAI_TOKEN = os.environ.get("CIVITAI_TOKEN", "")  # Токен Civitai для приватных/ограниченных ссылок.
HF_TOKEN = os.environ.get("HF_TOKEN", "")  # Токен Hugging Face для gated/ограниченных ссылок.

CUSTOM_NODE_REPOS = [  # Список custom nodes, обязательных по ТЗ.
    "https://github.com/ltdrdata/ComfyUI-Manager",
    "https://github.com/cubiq/ComfyUI_IPAdapter_plus",
    "https://github.com/Fannovel16/comfyui_controlnet_aux",
]

CHECKPOINT_URLS = []  # Список URL базовых моделей (checkpoints). Добавляйте ваши ссылки.
LORA_URLS = []  # Список URL LoRA моделей.
UPSCALER_URLS = []  # Список URL upscaler моделей.
VAE_URLS = []  # Список URL VAE файлов.
CONTROLNET_URLS = []  # Список URL ControlNet моделей.
IPADAPTER_URLS = ["https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin"]  # Обязательный файл FaceID PlusV2.


def run_cmd(cmd, check=False):
    """Функция запускает команду и печатает её для прозрачности выполнения."""
    print("[CMD]", " ".join(str(x) for x in cmd))
    return subprocess.run(cmd, check=check)


def with_token_if_needed(url):
    """Функция добавляет токен в URL для HF/Civitai, если токен есть и он ещё не добавлен."""
    if "huggingface.co" in url and HF_TOKEN:
        separator = "&" if "?" in url else "?"  # separator выбирает корректный разделитель query-параметров.
        return f"{url}{separator}token={HF_TOKEN}"
    if "civitai.com" in url and CIVITAI_TOKEN and "token=" not in url:
        separator = "&" if "?" in url else "?"
        return f"{url}{separator}token={CIVITAI_TOKEN}"
    return url


def file_name_from_url(url):
    """Функция извлекает имя файла из URL-адреса."""
    return url.split("?")[0].rstrip("/").split("/")[-1]


def ensure_custom_node(repo_url):
    """Функция клонирует custom node или обновляет его, если репозиторий уже существует."""
    repo_name = repo_url.rstrip("/").split("/")[-1]  # repo_name — имя репозитория из URL.
    repo_dir = Path(CUSTOM_NODES_DIR) / repo_name  # repo_dir — целевой путь репозитория в папке custom_nodes.
    if (repo_dir / ".git").exists():
        run_cmd(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=False)
    else:
        run_cmd(["git", "clone", repo_url, str(repo_dir)], check=False)


def download_one(url, target_dir):
    """Функция скачивает один файл в нужную папку и возвращает статус операции."""
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    safe_url = with_token_if_needed(url)  # safe_url — URL, в который при необходимости подставлен токен.
    file_name = file_name_from_url(url)  # file_name — имя локального файла на диске.
    output_path = target_dir / file_name  # output_path — итоговый путь загруженного файла.
    if output_path.exists() and output_path.stat().st_size >= MIN_VALID_FILE_BYTES:
        return (url, "skip", str(output_path))
    run_cmd(["aria2c", safe_url, "-x", "8", "-s", "8", "-k", "1M", "--allow-overwrite=true", "-d", str(target_dir), "-o", file_name], check=False)
    if output_path.exists() and output_path.stat().st_size >= MIN_VALID_FILE_BYTES:
        return (url, "ok", str(output_path))
    return (url, "fail", str(output_path))


for repo in CUSTOM_NODE_REPOS:
    ensure_custom_node(repo)  # Сначала готовим custom nodes (зависимости должны быть готовы после ячейки 3).

# Формируем общий список задач на скачивание с приоритетами.
download_jobs = []  # download_jobs хранит кортежи (url, target_dir, priority).
for url in CHECKPOINT_URLS:
    download_jobs.append((url, CHECKPOINTS_DIR, 1))
for url in CONTROLNET_URLS:
    download_jobs.append((url, CONTROLNET_DIR, 1))
for url in IPADAPTER_URLS:
    download_jobs.append((url, IPADAPTER_DIR, 1))
for url in LORA_URLS:
    download_jobs.append((url, LORA_DIR, 2))
for url in VAE_URLS:
    download_jobs.append((url, VAE_DIR, 3))
for url in UPSCALER_URLS:
    download_jobs.append((url, UPSCALER_DIR, 4))

download_jobs = sorted(download_jobs, key=lambda item: item[2])  # Сортировка: сначала тяжёлые типы файлов (меньший priority).
results = []  # Список результатов всех задач скачивания.

with ThreadPoolExecutor(max_workers=MAX_PARALLEL_DOWNLOADS) as pool:
    future_map = {pool.submit(download_one, job[0], job[1]): job for job in download_jobs}  # future_map связывает future с исходной задачей.
    for future in as_completed(future_map):
        results.append(future.result())

ok_count = sum(1 for _, status, _ in results if status == "ok")  # Количество успешных загрузок.
skip_count = sum(1 for _, status, _ in results if status == "skip")  # Количество пропусков (файлы уже есть).
fail_count = sum(1 for _, status, _ in results if status == "fail")  # Количество неуспешных загрузок.
print(f"✅ Загрузки завершены: ok={ok_count}, skip={skip_count}, fail={fail_count}")
if fail_count:
    print("⚠️ Есть неуспешные загрузки, проверьте ссылки/токены.")


In [ ]:
# ЯЧЕЙКА 5: запуск ComfyUI в двух режимах по выбору пользователя.
import os  # Импорт os нужен для токена и переменных окружения.
import subprocess  # Импорт subprocess нужен для запуска процессов ComfyUI/cloudflared.
import time  # Импорт time нужен для задержек при ожидании URL туннеля.
import re  # Импорт re нужен для поиска публичного URL в логах cloudflared.
from pathlib import Path  # Импорт Path нужен для проверки существования main.py.

comfy_main = Path(COMFY_DIR) / "main.py"  # Путь к файлу запуска ComfyUI.
if not comfy_main.exists():
    raise FileNotFoundError(f"Не найден файл запуска ComfyUI: {comfy_main}")

print("1) вариант запуска: локальный режим без доступа извне (для SSH-туннеля на Vast.ai)")
print("2) вариант запуска: временный сайт через tunnel + GRADIO_TOKEN")
launch_mode = input("Введите режим запуска (1/2): ").strip()  # launch_mode хранит выбор пользователя.
if launch_mode not in {"1", "2"}:
    raise ValueError("Нужно ввести только 1 или 2")

comfy_cmd = ["python", str(comfy_main), "--port", "8188"]  # Базовая команда запуска ComfyUI.

if launch_mode == "1":
    comfy_cmd += ["--listen", "127.0.0.1"]  # В режиме 1 ComfyUI доступен только локально.
    print("Запуск локального режима ComfyUI. Используйте SSH-туннель к 127.0.0.1:8188")
    subprocess.Popen(comfy_cmd)
else:
    comfy_cmd += ["--listen", "0.0.0.0"]  # В режиме 2 ComfyUI слушает внешний интерфейс для tunnel.
    print("Запуск ComfyUI для временного внешнего доступа...")
    subprocess.Popen(comfy_cmd)
    time.sleep(3)

    cloudflared_exists = subprocess.run(["bash", "-lc", "command -v cloudflared >/dev/null"], check=False).returncode == 0  # Флаг наличия cloudflared.
    if not cloudflared_exists:
        subprocess.run(["bash", "-lc", "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared"], check=False)

    tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)  # tunnel_proc — процесс временного туннеля.

    public_url = None  # Переменная public_url хранит найденную публичную ссылку tunnel.
    start_time = time.time()  # Переменная start_time нужна для ограничения времени ожидания.
    while time.time() - start_time < 30:
        line = tunnel_proc.stdout.readline()
        if not line:
            continue
        print(line.rstrip())
        match = re.search(r"https://[a-zA-Z0-9\-\.]+trycloudflare.com", line)  # match хранит найденный URL из строки логов.
        if match:
            public_url = match.group(0)
            break

    access_token = os.environ.get("GRADIO_TOKEN", "set-GRADIO_TOKEN")  # access_token берётся из GRADIO_TOKEN для простого контроля доступа ссылкой.
    if public_url:
        print("✅ Временный публичный URL создан:")
        print(public_url)
        print("🔐 Открывайте ссылку с token-параметром:")
        print(f"{public_url}/?token={access_token}")
    else:
        print("⚠️ Не удалось получить публичный URL tunnel за отведённое время.")


In [ ]:
# ЯЧЕЙКА 6: остановка (убийство) процессов ComfyUI.
import subprocess  # Импорт subprocess нужен для выполнения команды завершения процесса.

kill_cmd = ["bash", "-lc", "pkill -f 'ComfyUI/main.py' || true"]  # Команда завершает процессы, где есть путь ComfyUI/main.py.
subprocess.run(kill_cmd, check=False)  # Выполняем остановку; если процесса нет, ошибка подавляется через `|| true`.
print("✅ Процессы ComfyUI остановлены (если были запущены).")
